# Search the Bundestag energy papers

Type a question, press **Search**, and you get the text passages from Bundestag papers that match it best, with a link to the PDF.

**How to use:** run all cells once (menu *Run → Run All*). Loading takes about 10 seconds. After that, you can search as often as you like.

**First time only:** build the index (see the scripts folder) and start the notebook from the project folder:

```
.venv/bin/pip install -r requirements-notebook.txt
.venv/bin/jupyter notebook notebooks/search.ipynb
```

In [ ]:
# Load the search index and the embedding model. Takes about 10 seconds.
import os
import sys
from pathlib import Path

# Work from the project folder, so "data/chroma" and the code in "src" are found.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from energiewende.index import search, store

index = store.load_index()
print(store.count_chunks(), "text passages ready to search")

In [ ]:
# The search box.
import html

import ipywidgets as widgets
from IPython.display import HTML, display

question_box = widgets.Text(
    placeholder="z.B. Wie soll der Ausbau von Wasserstoff gefördert werden?",
    layout=widgets.Layout(width="80%"),
)
results_slider = widgets.IntSlider(value=4, min=1, max=10, description="Results")
search_button = widgets.Button(description="Search", button_style="primary")
output = widgets.Output()


def show_hit(hit):
    """One result as a small card: title, source, and the matching passage."""
    passage = html.escape(hit["text"][:700])
    display(HTML(f"""
    <div style="border-left: 4px solid #2a7ae2; padding: 4px 12px; margin: 12px 0;">
      <b>{html.escape(hit["title"][:200])}</b><br>
      <small>Drucksache {hit["number"]} · {hit["date"]} · score {hit["score"]} ·
      <a href="{hit["url"]}" target="_blank">PDF</a></small>
      <p style="white-space: pre-wrap;">{passage} …</p>
    </div>"""))


def on_search(_):
    output.clear_output()
    question = question_box.value.strip()
    if not question:
        return
    with output:
        for hit in search.search(index, question, k=results_slider.value):
            show_hit(hit)


search_button.on_click(on_search)
display(widgets.HBox([question_box, search_button]), results_slider, output)